# 09 非レバ3戦略・VT3戦略の比較と収益源の分解
## 1. 目的と比較基準
同じデータで、①リスク資産100%の3戦略、②現在のVT3戦略、③②の収益源を比較します。
基準は**金利0・費用後の非レバ等ウェイト**です。現在のVTは目標10%、月次目標上限1.5倍。
無上限も保存しています。金利0%が基準で、2/3%、4/6%は預金/借入の感応度です。
[定義](../資料/09_戦略比較と収益分解の定義.md) /
[レポート](../outputs/reports/戦略比較と収益分解_分析レポート.md)。

## 2. 理論・恒等式
期首の実保有倍率L、リスク資産内構成qから $a_s=q_s^\top r$, $a_b=q_{EW}^\top r$ とすると、
$$r_{VT}-r_{EW}=(a_s-a_b)+(L-1)a_s+I_{cash}/NAV-I_{borrow}/NAV-\Delta TC.$$
さらに $(L-1)a_s=(L-1)a_b+(L-1)(a_s-a_b)$。
最後の交互作用は倍率効果の内数です。二重に合計しません。

日次寄与の平均×252は年率算術差です。複数期間では
$$k_t=\frac{\log(1+r_{VT,t})-\log(1+r_{EW,t})}{r_{VT,t}-r_{EW,t}}$$
で連結し、252/n×Σk_t c_j,tを年率対数差への寄与として出します。同リターン時は1/(1+r_EW)。
これらの単位をCAGR差と混同しません。Sharpe・最大DDそのものの要因分解ではありません。

## 3. データ / 4. 条件
既存のSPY/TLT/GLD調整終値スナップショット、USD、2006-01-03〜2026-10-29、5239営業日。
252日推定、月末判断→翌終値約定、片道5bps。未来の実現ボラは売買に使いません。
後付けの追加研究で、独立した将来OOS・因果実験ではありません。元データは更新しません。

In [ ]:
from pathlib import Path
import os, json, inspect
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Code
from quant_ai.return_attribution import (COMPONENTS, load_attribution_study,
    daily_attribution, annual_contributions, cagr_bridge, actual_payments)
from quant_ai.risk_allocation import monthly_targets
from quant_ai.volatility_target import volatility_schedule
from quant_ai.leveraged_backtest import leveraged_backtest
from quant_ai.attribution_outputs import attribution_figures
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p/'src/quant_ai').exists())
os.chdir(ROOT)
prices=pd.read_parquet(ROOT/'data/processed/spy_tlt_gld_adjusted.parquet')
study=load_attribution_study(ROOT)
display(pd.Series(study.provenance))


## 5. 実装と実際の再計算
[分解関数](../src/quant_ai/return_attribution.py)をF12で追えます。ここでも実装を表示します。
CAGRの橋渡しでは、費用0の配分差→費用0の倍率差→追加費用→借入→預金の比較経路を使います。
各効果は独立の「要因別CAGR」ではなく、合計が最終CAGR差へ一致するよう定義した差です。

In [ ]:
display(Code(inspect.getsource(daily_attribution),language='python'))
display(Code(inspect.getsource(cagr_bridge),language='python'))
base,base_audit=monthly_targets(prices,'min_variance')
equal,equal_audit=monthly_targets(prices,'equal_weight')
targets,audit=volatility_schedule(prices,'min_variance',leverage_cap=1.5,
                                 base_targets=base,base_audit=base_audit)
sample=prices.loc[base.index[0]:]
bg=leveraged_backtest(sample,equal,cost_bps=0)
bn=leveraged_backtest(sample,equal)
rg=leveraged_backtest(sample,base,cost_bps=0)
raw=leveraged_backtest(sample,base)
vg=leveraged_backtest(sample,targets,cost_bps=0)
vz=leveraged_backtest(sample,targets)
vb=leveraged_backtest(sample,targets,borrow_rate=.03)
vf=leveraged_backtest(sample,targets,cash_rate=.02,borrow_rate=.03)
bridge=cagr_bridge(bg,bn,rg,vg,vz,vb,vf)
saved=study.bridges.loc['cap15_rates23_min_variance']
np.testing.assert_allclose([bridge[c] for c in COMPONENTS],saved[list(COMPONENTS)].to_numpy(float),atol=1e-12)
pieces=daily_attribution(prices,vf,bn,raw)
linked=annual_contributions(pieces)
np.testing.assert_allclose(pieces[list(COMPONENTS)].sum(axis=1),pieces.active_return,atol=1e-12)
display(pd.Series(bridge))
display(pieces.head())


## 6. 結果と図
最初に金利0の非レバ3戦略・VT3戦略を見ます。金利感応度を混ぜて基準を置き換えません。
図03の上段は基準0%、下段は預金2%/借入3%です。比率上限は月次目標で、月中には超過し得ます。

In [ ]:
primary_ids=[f'raw_{m}' for m in ['equal_weight','inverse_vol','min_variance']]+[
    f'cap15_rates0_{m}' for m in ['equal_weight','inverse_vol','min_variance']]
display(study.performance.loc[primary_ids,['annual_return','volatility','max_drawdown','sharpe','mean_held_leverage']])
display(study.bridges[study.bridges['mode'].eq('cap15')])
figures=attribution_figures(study,ROOT)
for name,fig in figures.items():
    print(name)
    display(fig)
    plt.close(fig)


## 7. 統計的解釈
この分解は会計上の恒等式と、定義した比較経路の差です。因果的な仮説検定ではありません。
全期間や配分を見た後に設計したため、独立なスキル・将来の優位性を証明しません。
算術年率寄与・対数年率寄与・CAGR橋渡しは異なる量なので、同じ値になる必要はありません。

In [ ]:
display(linked)
display(actual_payments(vf,prices))
expected_log=(np.log1p(vf.daily.net_return).mean()-np.log1p(bn.daily.net_return).mean())*252
print('年率対数差:',expected_log)
print('寄与の合計:',linked.loc[list(COMPONENTS),'annual_log_contribution'].sum())
print('CAGR差:',bridge['total_cagr_difference'])
print('実支払借入費用（初期1USDあたり）:',vf.daily.funding_cost.sum())
display(pd.Series(study.audit))


## 8. 金融工学・運用上の解釈
Lは予測ボラに応じて配分と同時に変わります。倍率項は平均投資量だけでなく、タイミングや
配分との交互作用も含みます。比較順序や基準を変えると寄与も変わり得ます。
実際の借入支払額と、借入なし経路との差は、単位・NAV経路・複利が異なるため一致しません。
現在の金利0ケースで調達費用0となる点を、現実にも借入無料という意味で解釈しません。

## 9. 限界と追加検証
レジーム・資産選択・借入条件・約定ずれ・税・市場インパクトの制約は残ります。
この分解から直ちに投資信託として採用可とは判断しません。別基準・別期間・独立将来での検証が必要です。
全計算は `python scripts/run_return_attribution.py`、図と報告だけなら `--render-only`。
実データの出力と、閉形式・境界を確認する合成テストを区別してください。